# Ledoit–Wolf shrinkage, from scratch

_Build the shrinkage estimator and measure its out-of-sample payoff._

**pyportfolios.com** · [/research/ledoit-wolf-shrinkage](https://pyportfolios.com/research/ledoit-wolf-shrinkage)

> Runs on numpy + pandas + scipy + matplotlib. Synthetic, seeded data — illustrative, not investment advice.

## The estimator (matches sklearn.covariance.LedoitWolf)

In [ ]:
import numpy as np

def ledoit_wolf(X):
    """X: (T, N) returns, rows = observations. Returns (shrunk_cov, delta)."""
    T, N = X.shape
    X = X - X.mean(axis=0)
    S = X.T @ X / T
    mu = np.trace(S) / N
    F = mu * np.eye(N)
    d2 = ((S - F) ** 2).sum() / N
    X2 = X ** 2
    b2 = ((X2.T @ X2) / T - S ** 2).sum() / (N * T)
    b2 = min(b2, d2)
    delta = b2 / d2 if d2 > 0 else 0.0
    return (1 - delta) * S + delta * F, delta

## Out-of-sample: minimum-variance portfolio, sample vs shrinkage

In [ ]:
rng = np.random.default_rng(0)
T, N, k = 320, 40, 3
beta = rng.normal(size=(N, k))
factors = rng.normal(size=(T, k))
idio = rng.normal(size=(T, N)) * 0.6
X = (factors @ beta.T + idio) / 100        # daily-ish returns with factor structure

def min_var(cov):
    inv = np.linalg.pinv(cov)
    w = inv @ np.ones(cov.shape[0])
    return w / w.sum()

split = 220
Xtr, Xte = X[:split], X[split:]
S = np.cov(Xtr, rowvar=False)
LW, delta = ledoit_wolf(Xtr)
print(f"shrinkage intensity delta = {delta:.3f}\n")

for name, M in [("sample", S), ("ledoit-wolf", LW)]:
    w = min_var(M)
    oos_vol = np.std(Xte @ w) * np.sqrt(252)
    print(f"{name:>12}:  OOS vol {oos_vol:6.2%}   max |w| {np.abs(w).max():5.2f}")